# Jupiter Notebook para explorar los datos que hemos subido a S3 y ver su contenido

In [2]:
import sys
from pathlib import Path
import io
import pyarrow.parquet as pq
import pandas as pd

# 1. Asegurar visibilidad de 'src' independientemente de donde este el notebook
# Si el notebook está en 'src/notebooks', el directorio 'src' es el padre de notebooks
SRC_PATH = str(Path.cwd().resolve()) if Path.cwd().name == "src" else str(Path.cwd().resolve().parent)
if not Path(SRC_PATH).joinpath("dependencies").exists():
    # Fallback si se ejecuta desde la raiz del repositorio
    SRC_PATH = str(Path.cwd().resolve() / "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# 2. Inyección de dependencias de infraestructura
from dependencies.dependencies import get_storage_repository, get_compactor

storage = get_storage_repository()
compactor = get_compactor()

print(f" Infraestructura conectada a S3: [{storage.bucket_name}]")

 Infraestructura conectada a S3: [tfm-master-prediccion-bancos-usa]


Descargamos los dos datasets de 8-K y 10-Q/10-K desde Cloudflare R2 y los guardamos en un dataframe para explorar los datos en Pandas.

In [5]:
# 1. Descargar bytes de forma asíncrona (usando await) y sin barra inicial
raw_bytes_10k = await storage.get(key="raw/sec/10k-10q-2010-2023.parquet")
raw_bytes_8k = await storage.get(key="raw/sec/8k-2010-2023.parquet")

# 2. Descompactar a DataFrames de Pandas
df_10k_10q = pd.DataFrame(compactor.decompact(raw_bytes_10k))
df_8k = pd.DataFrame(compactor.decompact(raw_bytes_8k))

print(f" 10-K / 10-Q cargado: {len(df_10k_10q):,} registros.")
print(f" 8-K cargado: {len(df_8k):,} registros.")

 INFO     | 2026-10-22 09:12:59.314816 | S3 Storage Repository | Abriendo canal persistente con el almacenamiento de objetos S3 (Bootstrap Phase)...
 INFO     | 2026-10-22 09:12:59.678470 | S3 Storage Repository | Conectado a S3 con éxito. Bucket por defecto: [tfm-master-prediccion-bancos-usa] | Endpoint: https://ab02a9e85571f8564208e420a50be80a.eu.r2.cloudflarestorage.com/
 INFO     | 2026-10-22 09:13:01.086684 | ParquetCompactor | Descompactados 336 registros desde buffer Parquet (18,625 bytes).
 INFO     | 2026-10-22 09:13:01.137640 | ParquetCompactor | Descompactados 5,461 registros desde buffer Parquet (1,178,960 bytes).
 10-K / 10-Q cargado: 336 registros.
 8-K cargado: 5,461 registros.


In [7]:
df_10k_10q.head()

,ticker,company_name,cik,form,filing_date,report_date,acceptance_date_time,accession_number,items,primary_document,primary_doc_description,document_url,index_url
0,JPM,JPMorgan Chase & Co.,19617,10-Q,2023-11-01,2023-09-30,2023-11-02T00:29:55.000Z,0000019617-23-000524,,jpm-20230930.htm,10-Q,https://www.sec.gov/Archives/edgar/data/19617/...,https://www.sec.gov/Archives/edgar/data/19617/...
1,JPM,JPMorgan Chase & Co.,19617,10-Q,2023-08-03,2023-06-30,2023-08-04T00:41:34.000Z,0000019617-23-000432,,jpm-20230630.htm,10-Q,https://www.sec.gov/Archives/edgar/data/19617/...,https://www.sec.gov/Archives/edgar/data/19617/...
2,JPM,JPMorgan Chase & Co.,19617,10-Q,2023-05-03,2023-03-31,2023-05-04T00:10:19.000Z,0000019617-23-000310,,jpm-20230331.htm,10-Q,https://www.sec.gov/Archives/edgar/data/19617/...,https://www.sec.gov/Archives/edgar/data/19617/...
3,JPM,JPMorgan Chase & Co.,19617,10-K,2023-02-21,2022-12-31,2023-02-22T02:13:55.000Z,0000019617-23-000231,,jpm-20221231.htm,10-K,https://www.sec.gov/Archives/edgar/data/19617/...,https://www.sec.gov/Archives/edgar/data/19617/...
4,JPM,JPMorgan Chase & Co.,19617,10-Q,2022-11-03,2022-09-30,2022-11-04T00:13:37.000Z,0000019617-22-000492,,jpm-20220930.htm,10-Q,https://www.sec.gov/Archives/edgar/data/19617/...,https://www.sec.gov/Archives/edgar/data/19617/...


In [8]:
df_8k.head()

,ticker,company_name,cik,form,filing_date,accession_number,item,item_description,content,document_url
0,JPM,JPMorgan Chase & Co.,19617,8-K,2026-10-21,0000019617-23-000063,Item 2.02,Results of Operations and Financial ConditionO...,(“JPMorgan Chase” or the “Firm”) reported 2022...,https://www.sec.gov/Archives/edgar/data/19617/...
1,JPM,JPMorgan Chase & Co.,19617,8-K,2026-10-21,0000019617-23-000063,Item 9.01,Financial Statements and Exhibits(d) ExhibitsE...,Description of Exhibit99.1JPMorgan Chase & Co....,https://www.sec.gov/Archives/edgar/data/19617/...
2,JPM,JPMorgan Chase & Co.,19617,8-K,2026-10-21,0001193125-19-287311,Item 3.03,Material Modification to Rights of Security Ho...,"(the “Company”) issued 90,000 shares (the “Sha...",https://www.sec.gov/Archives/edgar/data/19617/...
3,JPM,JPMorgan Chase & Co.,19617,8-K,2026-10-21,0001193125-19-287311,Item 5.03,Amendments to Articles of Incorporation or Byl...,"On November 6, 2019, the Company filed a Certi...",https://www.sec.gov/Archives/edgar/data/19617/...
4,JPM,JPMorgan Chase & Co.,19617,8-K,2026-10-21,0001193125-19-287311,Item 8.01,Other Events,"On November 7, 2019, the Company completed the...",https://www.sec.gov/Archives/edgar/data/19617/...
